# Required Setup — Neo4j Credentials

**Run this notebook once before starting any other notebooks.**

This notebook stores your Neo4j Aura connection details as Databricks secrets
so all subsequent notebooks (`01_neo4j_ingest`, `02_gds_algorithms`, `03_pull_and_model`)
can connect without re-entering credentials.

The Delta Lake tables (accounts, account_labels, merchants, transactions, account_links)
were already loaded by your workshop admin.

### Prerequisites

- A running **Neo4j Aura** instance with connection details ready
- A **Dedicated** compute cluster with the Neo4j Spark Connector installed:
  - Access mode: **Dedicated** (required for the Spark Connector)
  - Runtime: 13.3 LTS or higher
  - Maven library: `org.neo4j:neo4j-connector-apache-spark_2.12:5.3.1_for_spark_3`

## Step 1: Enter Your Neo4j Connection Details

Fill in the three widgets at the top of this notebook with your Neo4j Aura
connection information. These values will be stored as Databricks secrets
in scope `neo4j-graph-engineering` with keys `uri`, `username`, and `password`.

In [ ]:
NEO4J_URI      = 'NEO4J_URI'
NEO4J_USERNAME = 'neo4j_username'
NEO4J_PASSWORD = 'neo4j_password'

if NEO4J_URI == 'NEO4J_URI' or NEO4J_PASSWORD == 'neo4j_password':
    raise ValueError(
        'Please replace the NEO4J_URI, NEO4J_USERNAME, and NEO4J_PASSWORD placeholders before running.'
    )

## Step 2: Store Credentials and Verify Connection

The cells below create the secret scope `neo4j-graph-engineering` (if it does not
already exist), store your credentials, then verify the connection is live.

In [ ]:
from databricks.sdk import WorkspaceClient

SCOPE = 'neo4j-graph-engineering'
w = WorkspaceClient()

print(f'Creating secret scope: {SCOPE}')
try:
    w.secrets.create_scope(scope=SCOPE)
    print(f'  [OK] Scope {SCOPE!r} created')
except Exception as e:
    if 'already exists' in str(e).lower():
        print(f'  [OK] Scope {SCOPE!r} already exists')
    else:
        raise

for key, value in {'uri': NEO4J_URI, 'username': NEO4J_USERNAME, 'password': NEO4J_PASSWORD}.items():
    w.secrets.put_secret(scope=SCOPE, key=key, string_value=value)
    print(f'  [OK] {key} stored')

In [ ]:
%pip install neo4j --quiet

In [ ]:
from neo4j import GraphDatabase

print(f'Verifying Neo4j connection: {NEO4J_URI}')
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    with driver.session(database='neo4j') as session:
        record = session.run("RETURN 'Connected' AS status").single()
        print(f'  [OK] Neo4j responded: {record["status"]}')
    driver.close()
except Exception as e:
    print(f'  [FAIL] Could not connect to Neo4j: {e}')
    print('  Check that the URI starts with neo4j+s:// for Aura and credentials are correct.')
    raise

## Next Steps

If both steps above show `[OK]`, proceed to **`01_neo4j_ingest`** to push the
synthetic fraud dataset into Neo4j via the Spark Connector.